# 365 Probabilidades - Dia #129
## Qual a probabilidade de você perceber que estão mentindo para você?

**Tipo:** Comportamental
**Data de publicação:** 2026-10-20
**Ferramenta:** Python
**Decisão analisada:** Vale confiar no seu faro para mentira?
**Hashtag:** #365Probabilidades #Dia129

---

### 📖 A História

Todo mundo acha que pega a mentira. Pelo olhar que desvia, pela pausa antes da resposta,
pelo jeito de mexer na mão. Cada um tem o seu sinal e confia nele.

Fui ver se alguém já tinha medido, em gente de verdade julgando gente de verdade, quantas
mentiras passam.

---

### 📚 O Conceito

A pesquisa sobre detecção de mentira tem um desenho simples. Uma pessoa grava mensagens,
algumas verdadeiras e outras falsas. Outra pessoa assiste e diz, para cada uma, se é
mentira ou verdade. A proporção de acertos é a medida. Em 2006, dois psicólogos reuniram
tudo o que havia sido feito assim em meio século, 206 estudos e mais de 24 mil juízes.

O acerto tem dois lados, e eles não são iguais. Reconhecer uma verdade como verdade é
uma coisa. Perceber uma mentira como mentira é outra. A pergunta do dia é sobre a
segunda, e a resposta vem acompanhada do que a primeira revela: as pessoas acreditam
mais do que desconfiam.

---

### 🧮 O Modelo

Cinco camadas. A fonte central dá os acertos. As contas são nossas.

1. **A chance de perceber.** A fatia das mentiras percebidas e das verdades
   reconhecidas, com a variação entre estudos.
2. **Quando você desconfia.** Bayes com os dois acertos e a frequência de mentiras nas
   conversas: se você acha que é mentira, qual a chance de ser.
3. **A vantagem sobre a moeda.** O tamanho de efeito do acerto traduzido em duelos.
4. **Dez mentiras.** Monte Carlo de dez mentiras contadas a você: quantas você pega.
5. **As mentiras de um dia.** Conta de bolso com quantas mentiras uma pessoa conta por
   dia e quantas são descobertas, segundo quem mentiu.

**Fonte central:**
- Bond, C. F., Jr. & DePaulo, B. M. (2006). "Accuracy of deception judgments".
  *Personality and Social Psychology Review* 10(3), 214-234. Meta-análise de **206
  estudos**, 384 amostras de juízes, **24.483 juízes** julgando 6.651 mensagens de 4.435
  pessoas. Acerto geral 53,46% [53,31; 53,59]; mentiras percebidas **47,55%**; verdades
  reconhecidas 61,34%; mensagens julgadas verdadeiras 55,23% [54,99; 55,46]; desvio real
  do acerto entre estudos 4,52 pontos; d médio do acerto 0,40 (0,39 ponderado);
  especialistas contra leigos d = −0,025 [−0,105; 0,055]; só vídeo 50,52%, áudio 53,01%,
  audiovisual 54,03%. **Lido no texto completo.**

**Fontes de apoio:**
- DePaulo, B. M., Kashy, D. A., Kirkendol, S. E., Wyer, M. M. & Epstein, J. A. (1996).
  "Lying in everyday life". *Journal of Personality and Social Psychology* 70(5),
  979-995. Diário de 7 dias: **77 universitários** (1,96 mentiras por dia, 0,31 por
  interação) e **70 adultos da comunidade** (0,97 por dia, 0,20 por interação, mentira em
  cerca de uma de cada cinco interações). Mentiras descobertas, segundo quem mentiu:
  22,78% (universitários) e 14,85% (comunidade). Lido no texto completo.
- Serota, K. B., Levine, T. R. & Boster, F. J. (2010). "The prevalence of lying in
  America: three studies of self-reported lies". *Human Communication Research* 36,
  2-25. Pesquisa nacional com **1.000 adultos** (998 analisados): média de 1,65 mentiras
  por dia, mediana zero, **59,9% sem mentira nas últimas 24 horas**; metade das mentiras
  contadas por 5,3% das pessoas. Lido no texto completo.

**Nota metodológica.** O fator ×0,80 do projeto não se aplica à fonte central (acerto
medido em experimento). Aplica-se às proporções de autorrelato das fontes de apoio
(mentiras por dia, mentiras descobertas), que entram na camada 5 com e sem o fator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 42
SORTEIOS = 200_000
rng = np.random.default_rng(SEMENTE)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
RODAPE = 'Fonte: Bond & DePaulo 2006, PSPR 10(3) · DePaulo et al. 1996 · Serota et al. 2010 · cálculo próprio | #365Probabilidades'

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(x, casas=1):
    return vir(100 * x, casas) + "%"

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS DA LITERATURA ---
# Bond & DePaulo (2006), PSPR 10(3), 214-234
N_ESTUDOS, N_JUIZES, N_MENSAGENS = 206, 24_483, 6_651
ACERTO = 0.5346            # acerto geral, média ponderada
ACERTO_IC = (0.5331, 0.5359)
P_MENTIRA_PEGA = 0.4755    # mentiras percebidas como mentira
P_VERDADE_RECONHECIDA = 0.6134
P_JULGADO_VERDADE = 0.5523
DP_ENTRE_ESTUDOS = 0.0452  # desvio real do acerto entre estudos (sem o erro amostral)
D_ACERTO = 0.40            # tamanho de efeito médio do acerto
D_ESPECIALISTA = (-0.025, -0.105, 0.055)
ACERTO_MEIO = {'só vídeo': 0.5052, 'só áudio': 0.5301, 'áudio e vídeo': 0.5403}

# DePaulo et al. (1996), JPSP 70(5): diário de 7 dias
MENTIRAS_DIA = {'universitários': 1.96, 'comunidade': 0.97}
MENTIRA_POR_INTERACAO = {'universitários': 0.31, 'comunidade': 0.20}
DESCOBERTAS = {'universitários': 0.2278, 'comunidade': 0.1485}
# Serota, Levine & Boster (2010), HCR 36
SEROTA_N, SEROTA_MEDIA, SEROTA_ZERO, SEROTA_TOPO = 998, 1.65, 0.599, 0.053
FATOR = 0.80

print("=" * 72)
print("  DADOS · BOND & DEPAULO (2006)")
print("=" * 72)
print(f"\n  {N_ESTUDOS} estudos · {N_JUIZES:,} juízes · {N_MENSAGENS:,} mensagens".replace(",", "."))
print(f"  Acerto geral: {pct(ACERTO, 2)} [{pct(ACERTO_IC[0], 2)}; {pct(ACERTO_IC[1], 2)}]")
print(f"  Mentiras percebidas como mentira: {pct(P_MENTIRA_PEGA, 2)}")
print(f"  Verdades reconhecidas como verdade: {pct(P_VERDADE_RECONHECIDA, 2)}")
print(f"  Mensagens julgadas verdadeiras: {pct(P_JULGADO_VERDADE, 2)}")
print(f"  Variação real do acerto entre estudos: {vir(DP_ENTRE_ESTUDOS * 100)} pontos")
print(f"  Especialistas contra leigos: d = {D_ESPECIALISTA[0]} [{D_ESPECIALISTA[1]}; {D_ESPECIALISTA[2]}]")
for k, v in ACERTO_MEIO.items():
    print(f"    {k:>14}: {pct(v, 1)}")
print("\n  DePaulo et al. (1996), diário de 7 dias:")
for g in MENTIRAS_DIA:
    print(f"    {g:>14}: {vir(MENTIRAS_DIA[g], 2)} mentiras por dia · {vir(MENTIRA_POR_INTERACAO[g], 2)} por interação · "
          f"{pct(DESCOBERTAS[g])} descobertas")
print(f"\n  Serota et al. (2010), {SEROTA_N} adultos: média {vir(SEROTA_MEDIA, 2)} por dia, mediana zero, "
      f"{pct(SEROTA_ZERO)} sem mentira em 24 h, metade das mentiras de {pct(SEROTA_TOPO)} das pessoas")
print(f"\n  ×0,80: não na fonte central; sim nas proporções de autorrelato das fontes de apoio.")
print("=" * 72)

In [ ]:
# --- O MODELO ---

# ── CAMADA 1 · a chance de perceber ──
# Variação entre estudos: um estudo novo tem acerto ~ Normal(53,46; 4,52). Transporto a mesma
# variação para o acerto nas mentiras, premissa declarada.
novo = stats.norm(P_MENTIRA_PEGA, DP_ENTRE_ESTUDOS)
print("=" * 72)
print("  CAMADA 1 · A CHANCE DE PERCEBER")
print("=" * 72)
print(f"\n  Quando alguém mente, a chance de você perceber é de {pct(P_MENTIRA_PEGA)}.")
print(f"  Quando alguém fala a verdade, a chance de você acreditar é de {pct(P_VERDADE_RECONHECIDA)}.")
print(f"  Num estudo novo, o acerto nas mentiras fica em 95% das vezes entre {pct(novo.ppf(0.025))} e {pct(novo.ppf(0.975))}.")
print(f"  P(um estudo novo passar de 50% nas mentiras) = {pct(1 - novo.cdf(0.5))}")
print(f"  Especialistas: d = {D_ESPECIALISTA[0]}, intervalo que inclui zero. Policial e leigo erram igual.")
print("  Premissa: a variação entre estudos do acerto geral vale para o acerto nas mentiras.")

# ── CAMADA 2 · quando você desconfia (Bayes) ──
sens = P_MENTIRA_PEGA            # P(desconfia | mentira)
esp = P_VERDADE_RECONHECIDA      # P(acredita | verdade)
print("\n" + "=" * 72)
print("  CAMADA 2 · QUANDO VOCÊ DESCONFIA, QUAL A CHANCE DE SER MENTIRA?")
print("=" * 72)
print(f"\n  P(desconfiar | mentira) = {pct(sens)} · P(desconfiar | verdade) = {pct(1 - esp)}")
print(f"\n  {'mentiras na conversa':>22} {'desconfiou e era mentira':>26} {'acreditou e era verdade':>25}")
for p in (0.05, 0.10, 0.20, 0.31, 0.50):
    vpp = p * sens / (p * sens + (1 - p) * (1 - esp))
    vpn = (1 - p) * esp / ((1 - p) * esp + p * (1 - sens))
    marca = '  ← comunidade (DePaulo 1996)' if p == 0.20 else ('  ← universitários' if p == 0.31 else '')
    print(f"  {pct(p, 0):>22} {pct(vpp):>26} {pct(vpn):>25}{marca}")
p = 0.20
VPP = p * sens / (p * sens + (1 - p) * (1 - esp))
print(f"\n  Com mentira em uma de cada cinco falas, quando você desconfia, é mentira em {pct(VPP)} das vezes.")
print(f"  Em {pct(1 - VPP)} das vezes você desconfiou de quem falava a verdade.")

# ── CAMADA 3 · a vantagem sobre a moeda ──
sup = stats.norm.cdf(D_ACERTO / np.sqrt(2))
print("\n" + "=" * 72)
print("  CAMADA 3 · A VANTAGEM SOBRE A MOEDA")
print("=" * 72)
print(f"\n  d = {D_ACERTO}: em cem duelos entre uma mensagem verdadeira e uma falsa, você aponta")
print(f"  a falsa como mais suspeita em {vir(sup * 100, 0)} (Φ(d/√2) = {pct(sup)}).")
print(f"  A moeda acerta 50. Você acerta {pct(ACERTO)}. A vantagem existe e é de {vir((ACERTO - 0.5) * 100)} pontos.")

# ── CAMADA 4 · dez mentiras ──
N_MENT = 10
pegas = rng.binomial(N_MENT, P_MENTIRA_PEGA, SORTEIOS)
print("\n" + "=" * 72)
print(f"  CAMADA 4 · DEZ MENTIRAS CONTADAS A VOCÊ ({SORTEIOS:,} sorteios)".replace(",", "."))
print("=" * 72)
print(f"\n  Você percebe em média {vir(pegas.mean())} de {N_MENT}.")
print(f"  Percebe a metade ou mais: {pct((pegas >= 5).mean())} · nenhuma: {pct((pegas == 0).mean(), 2)} · todas: {pct((pegas == 10).mean(), 3)}")
print(f"  Oito ou mais: {pct((pegas >= 8).mean())}")
print("  Premissa: cada mentira é um julgamento independente, com o acerto médio da meta-análise.")

# ── CAMADA 5 · as mentiras de um dia (conta de bolso) ──
print("\n" + "=" * 72)
print("  CAMADA 5 · AS MENTIRAS DE UM DIA (conta de bolso, autorrelato)")
print("=" * 72)
for g in MENTIRAS_DIA:
    m = MENTIRAS_DIA[g]
    print(f"\n  {g}: {vir(m, 2)} mentiras por dia (com ×0,80: {vir(m * FATOR, 2)}) · "
          f"descobertas segundo quem mentiu: {pct(DESCOBERTAS[g])} (com ×0,80: {pct(DESCOBERTAS[g] * FATOR)})")
m = MENTIRAS_DIA['comunidade']
print(f"\n  Se as pessoas em volta mentem no ritmo da comunidade, numa semana chegam a você cerca de {vir(7 * m, 0)} mentiras")
print(f"  (com ×0,80, {vir(7 * m * FATOR, 0)}); você percebe {vir(7 * m * P_MENTIRA_PEGA)} (ou {vir(7 * m * FATOR * P_MENTIRA_PEGA)}).")
print(f"  Quem mentiu diz que só {pct(DESCOBERTAS['comunidade'])} foram descobertas. A diferença para os {pct(P_MENTIRA_PEGA)} do laboratório")
print(f"  é a distância entre desconfiar por dentro e dizer na cara.")
print(f"  Serota (2010): {pct(SEROTA_ZERO)} das pessoas não contaram mentira nenhuma em 24 horas; metade das mentiras vem de {pct(SEROTA_TOPO)}.")
print("  Premissa: você recebe tantas mentiras quanto a pessoa comum conta. Os dois diários são autorrelato.")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · os dois lados do acerto ══
fig1, ax1 = plt.subplots(figsize=(12, 7))
nomes = ['mentira percebida\ncomo mentira', 'verdade reconhecida\ncomo verdade', 'acerto geral', 'moeda']
vals = [P_MENTIRA_PEGA * 100, P_VERDADE_RECONHECIDA * 100, ACERTO * 100, 50]
bars = ax1.bar(nomes, vals, color=[VERMELHO, VERDE, DOURADO, CINZA], alpha=0.88, width=0.6)
for b, v in zip(bars, vals):
    ax1.text(b.get_x() + b.get_width() / 2, v + 0.8, pct(v / 100), ha='center', fontsize=13, fontweight='bold')
ax1.axhline(50, color=CINZA, linestyle='--', linewidth=1.5)
ax1.set_ylim(0, 72); ax1.set_ylabel('Acerto (%)', fontsize=12)
ax1.set_title('Perceber a mentira: 48 em cem. Acreditar na verdade: 61 em cem.\n'
              '206 estudos, 24.483 juízes, Bond & DePaulo 2006', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-129-grafico-01-os-dois-lados.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · quando você desconfia ══
fig2, ax2 = plt.subplots(figsize=(12, 7))
ps = np.linspace(0.01, 0.6, 300)
vpp = ps * sens / (ps * sens + (1 - ps) * (1 - esp))
ax2.plot(ps * 100, vpp * 100, color=VERMELHO, linewidth=3)
ax2.plot(ps * 100, ps * 100, color=CINZA, linestyle='--', linewidth=1.5, label='se você chutasse')
for p_, rot in [(0.20, 'comunidade: 1 em 5 falas'), (0.31, 'universitários: 1 em 3')]:
    v = p_ * sens / (p_ * sens + (1 - p_) * (1 - esp))
    ax2.scatter([p_ * 100], [v * 100], color=DOURADO, s=90, zorder=5)
    ax2.annotate(f'{rot}\ndesconfiou, era mentira em {pct(v)}', (p_ * 100, v * 100), xytext=(p_ * 100 + 3, v * 100 - 9),
                 fontsize=11, arrowprops=dict(arrowstyle='-', color=CINZA))
ax2.set_xlabel('Mentiras em cada cem falas que você ouve (%)', fontsize=12)
ax2.set_ylabel('Quando você desconfia, chance de ser mentira (%)', fontsize=12)
ax2.set_title('A desconfiança erra mais do que acerta\nBayes com os acertos da meta-análise; a frequência de mentiras vem do diário de 1996',
              fontsize=14, pad=16)
ax2.legend(frameon=False, fontsize=11, loc='upper left')
ax2.set_xlim(0, 60); ax2.set_ylim(0, 65)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-129-grafico-02-quando-voce-desconfia.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · dez mentiras ══
fig3, ax3 = plt.subplots(figsize=(12, 7))
k = np.arange(0, 11)
freq = np.bincount(pegas, minlength=11) / SORTEIOS
cores = [VERMELHO if i < 5 else VERDE for i in k]
ax3.bar(k, freq * 100, color=cores, alpha=0.88)
for i, f in zip(k, freq):
    if f > 0.004: ax3.text(i, f * 100 + 0.4, pct(f, 1), ha='center', fontsize=10)
ax3.set_xticks(k)
ax3.set_xlabel('Mentiras percebidas, de dez', fontsize=12); ax3.set_ylabel('Sorteios (%)', fontsize=12)
ax3.set_title(f'De dez mentiras contadas a você, você percebe {vir(pegas.mean())}. A metade ou mais em {pct((pegas >= 5).mean(), 0)} das vezes.\n'
              f"{SORTEIOS:,} sorteios, semente {SEMENTE}".replace(",", "."), fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-129-grafico-03-dez-mentiras.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · quem acerta mais ══
fig4, ax4 = plt.subplots(figsize=(12, 7))
nomes4 = ['só vídeo,\nsem som', 'só áudio', 'áudio e vídeo', 'todos os estudos']
vals4 = [ACERTO_MEIO['só vídeo'] * 100, ACERTO_MEIO['só áudio'] * 100, ACERTO_MEIO['áudio e vídeo'] * 100, ACERTO * 100]
cores4 = [VERMELHO, CINZA, CINZA, DOURADO]
bars = ax4.bar(nomes4, vals4, color=cores4, alpha=0.88, width=0.6)
for b, v in zip(bars, vals4):
    ax4.text(b.get_x() + b.get_width() / 2, v + 0.3, pct(v / 100), ha='center', fontsize=12, fontweight='bold')
ax4.axhline(50, color=CINZA, linestyle='--', linewidth=1.5); ax4.text(3.4, 50.3, 'moeda', color=CINZA, fontsize=11, ha='right')
ax4.set_ylim(45, 58); ax4.set_ylabel('Acerto geral (%)', fontsize=12)
ax4.text(0.5, 0.93, 'Especialistas (policiais, juízes) contra leigos: d = −0,025 [−0,105; 0,055]. Nenhuma diferença.',
         transform=ax4.transAxes, ha='center', fontsize=12, color=VERMELHO)
ax4.set_title('Ver sem ouvir é o pior jeito de julgar\nacerto geral por meio de apresentação, entre estudos', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-129-grafico-04-quem-acerta-mais.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**Quando alguém mente para você, a chance de você perceber é de 48%.** Menos do que a
moeda. Quando alguém fala a verdade, a chance de você acreditar é de 61%.

Esses dois números dizem a mesma coisa de dois jeitos: as pessoas acreditam mais do que
desconfiam. O detector que todo mundo acha que tem funciona a favor da confiança.

E quando a desconfiança aparece, ela erra mais do que acerta. Numa conversa em que uma
de cada cinco falas é mentira, se você desconfiou, era mesmo mentira em 24% das vezes.
Nas outras 76%, você desconfiou de quem falava a verdade.

De dez mentiras contadas a você, você percebe quatro ou cinco. Oito ou mais, em 4% das
vezes.

Especialistas, policiais e juízes acertam o mesmo que você. Ver o rosto sem ouvir a voz é
o pior jeito de julgar.

Pelo diário de quem mente, uma pessoa adulta conta cerca de uma mentira por dia, e diz
que só 15% delas foram descobertas. A maioria das pessoas passa o dia sem mentir, e
metade das mentiras vem de 5% delas.

*Quando foi a última vez que você desconfiou de alguém e estava errada?*

---

### ⚠️ Limitações do Modelo

- **É laboratório.** As mensagens são curtas, as pessoas se conhecem pouco e o juiz sabe
  que há mentira em jogo. Na vida há contexto, histórico e tempo, que a meta-análise não
  mede.
- **O acerto varia entre estudos.** O desvio real é de 4,5 pontos no acerto geral, e eu
  transporto a mesma variação para o acerto nas mentiras, premissa declarada.
- **A frequência de mentiras na conversa vem de um diário de 1996**, de 70 adultos de uma
  cidade americana, por autorrelato. A camada 2 mostra a conta para várias frequências.
- **As dez mentiras são julgamentos independentes**, com o acerto médio. A pessoa que
  mente mal e a que mente bem ficam iguais.
- **A conta de bolso da camada 5 é autorrelato dos dois lados**, com ×0,80 declarado, e
  supõe que você recebe tantas mentiras quanto a pessoa comum conta.
- A meta-análise é de 2006. Estudos posteriores sobre pistas verbais e entrevistas
  estratégicas não entram no dia.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*